# Channel Fusion / RGB-Encoding Feasibility — Visual Exploration

Goal: before touching the AKAZE+B-spline pipeline, visually compare candidate ways of
turning **CK + DAPI + AF** (3 of the 8 mIF channels) into the single detection image
that AKAZE/B-spline actually consume.

We look at three representations of the same three channels:

1. **Single-channel** — DAPI, AF, CK shown independently (log-normalized, what the
   current pipeline uses for CK only).
2. **Weighted grayscale fusion** — CK/DAPI/AF blended into one 8-bit image with
   tunable weights (this is the thing that actually changes what AKAZE sees).
3. **RGB encoding** — same three channels mapped to R/G/B for human QC. We also show
   what `cv2.cvtColor(..., COLOR_BGR2GRAY)` collapses that RGB composite back down to,
   to make concrete the point that AKAZE only ever sees a grayscale flattening — the
   RGB encoding is a visualization aid, not extra information for the detector.


In [ ]:
import os, sys
import numpy as np
import cv2
import tifffile
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = 'white'

# Make the parent dir importable so `config` resolves the same way the pipeline script does
current_dir = os.getcwd()
parent_dir  = os.path.dirname(current_dir)
sys.path.append(parent_dir)
try:
    import config
    DATASPACE = config.DATASPACE
except Exception as e:
    print("Could not import project `config` module - set DATASPACE manually below.")
    DATASPACE = None  


## 1. Load one slice

Point this at any core/slice you want to inspect. Defaults to the same
`TMA_Cores_Grouped_Rotate_Conformed/<core>` layout the registration script reads from.


In [ ]:
CORE_NAME   = "Core_01"     
SLICE_GLOB  = "*.tif"                   # adjust if your filename pattern differs
CHANNEL_NAMES = ['DAPI', 'CD31', 'GAP43', 'NFP', 'CD3', 'CD163', 'CK', 'AF']
DAPI_IDX, CK_IDX, AF_IDX = 0, 6, 7

import glob
input_folder = os.path.join(DATASPACE, "TMA_Cores_Grouped_Rotate_Conformed", CORE_NAME)
files = sorted(glob.glob(os.path.join(input_folder, SLICE_GLOB)))
print(f"Found {len(files)} files in {input_folder}")

SLICE_PATH = files[len(files)//2]   # pick a middle slice by default; change index/path as needed
print("Using:", SLICE_PATH)

arr = tifffile.imread(SLICE_PATH)
if arr.ndim == 2:
    arr = arr[np.newaxis]
elif arr.ndim == 3 and arr.shape[-1] < arr.shape[0]:
    arr = np.moveaxis(arr, -1, 0)
print("Volume shape (C, H, W):", arr.shape)

dapi = arr[DAPI_IDX].astype(np.float32)
ck   = arr[CK_IDX].astype(np.float32)
af   = arr[AF_IDX].astype(np.float32)


## 2. Helper functions

`prepare_single` mirrors `prepare_ck()` from the registration script (log-normalize
to 8-bit). `prepare_fused` is the proposed multi-channel extension — same per-channel
log-normalization, then a weighted blend into one 8-bit image.


In [ ]:
def prepare_single(img_arr, lo=0.1, hi=99.9):
    """Log-normalize a single channel to uint8 (same recipe as prepare_ck in the pipeline)."""
    img_float = img_arr.astype(np.float32)
    log_img   = np.log1p(img_float)
    p_lo, p_hi = np.percentile(log_img[::4, ::4], (lo, hi))
    norm = cv2.normalize(np.clip(log_img, p_lo, p_hi), None, 0, 255, cv2.NORM_MINMAX)
    return norm.astype(np.uint8)

def prepare_fused(channels, weights, lo=0.1, hi=99.9):
    """
    channels: list of 2D float arrays, e.g. [dapi, af, ck]
    weights:  same-length list of floats, relative contribution of each channel
    Returns a single uint8 fused detection image.
    """
    assert len(channels) == len(weights)
    norm_channels = [prepare_single(ch, lo, hi).astype(np.float32) for ch in channels]
    fused = sum(w * c for w, c in zip(weights, norm_channels)) / sum(weights)
    return np.clip(fused, 0, 255).astype(np.uint8)

def encode_rgb(channels, order=('R', 'G', 'B'), lo=0.1, hi=99.9):
    """
    Map up to 3 channels to R/G/B planes for visual QC.
    channels: list of up to 3 2D float arrays, in the same order as `order`.
    """
    norm_channels = [prepare_single(ch, lo, hi) for ch in channels]
    h, w = norm_channels[0].shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    slot = {'R': 0, 'G': 1, 'B': 2}
    for ch_img, tag in zip(norm_channels, order):
        rgb[..., slot[tag]] = ch_img
    return rgb


## 3. Single-channel representations

What AKAZE currently sees (CK only) vs. what DAPI/AF alone look like.


In [ ]:
dapi_u8 = prepare_single(dapi)
ck_u8   = prepare_single(ck)
af_u8   = prepare_single(af)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, [dapi_u8, ck_u8, af_u8], ['DAPI', 'CK (current pipeline input)', 'AF']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()


## 4. Weighted grayscale fusion (the actual proposal)

This is the image that would replace `fixed_log`/`moving_log` in `register_slice()`.
Try a few weight presets — CK-dominant (closest to current behavior), equal weight,
and DAPI-leaning (denser nuclear texture, useful where CK signal is weak/sparse).


In [ ]:
presets = {
    'CK-dominant (0.2 / 0.2 / 1.0)':   (0.2, 0.2, 1.0),   # dapi, af, ck
    'Equal weight (1 / 1 / 1)':        (1.0, 1.0, 1.0),
    'DAPI-leaning (1.0 / 0.3 / 0.6)':  (1.0, 0.3, 0.6),
}

fig, axes = plt.subplots(1, len(presets), figsize=(6 * len(presets), 6))
for ax, (label, w) in zip(axes, presets.items()):
    fused = prepare_fused([dapi, af, ck], weights=w)
    ax.imshow(fused, cmap='gray')
    ax.set_title(label)
    ax.axis('off')
plt.tight_layout()
plt.show()


## 5. RGB encoding (visual QC, not a different algorithm)

Each channel mapped to a color plane. We also show what this composite collapses to
under a standard grayscale conversion — this is what AKAZE would actually receive if
you fed it the RGB image directly, since OpenCV's feature detectors flatten color
internally before detecting.


In [ ]:
rgb = encode_rgb([dapi, af, ck], order=('R', 'G', 'B'))   # DAPI->R, AF->G, CK->B
rgb_as_gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)        # what AKAZE would actually see

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(rgb)
axes[0].set_title('RGB encoding (DAPI=R, AF=G, CK=B)\nfor human QC only')
axes[0].axis('off')
axes[1].imshow(rgb_as_gray, cmap='gray')
axes[1].set_title('Same composite, after cv2 grayscale flattening\n(this is what AKAZE actually detects on)')
axes[1].axis('off')
plt.tight_layout()
plt.show()


## 6. Side-by-side: RGB→gray vs. tuned weighted fusion

The point of this notebook: compare the *default* RGB→gray flattening (fixed luminance
weights, not tuned for IF data) against a *tuned* weighted fusion. If they look similar,
fusion weighting doesn't matter much. If the tuned version shows visibly more/cleaner
structure, it's worth plugging into `register_slice()`.


In [ ]:
tuned = prepare_fused([dapi, af, ck], weights=(1.0, 0.3, 0.6))  # match RGB order above for fair comparison

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(ck_u8, cmap='gray')
axes[0].set_title('Current: CK only')
axes[0].axis('off')

axes[1].imshow(rgb_as_gray, cmap='gray')
axes[1].set_title('RGB encoding -> default grayscale\n(untuned luminance weights)')
axes[1].axis('off')

axes[2].imshow(tuned, cmap='gray')
axes[2].set_title('Tuned weighted fusion\n(DAPI=1.0, AF=0.3, CK=0.6)')
axes[2].axis('off')
plt.tight_layout()
plt.show()


## 7. (Optional) Quick keypoint-count sanity check

A cheap proxy for "does fusion help AKAZE" before running the full pipeline: tissue-masked
AKAZE keypoint count on each candidate detection image. More keypoints (especially outside
the densest CK-only regions) is a positive signal, but isn't sufficient on its own — final
judgment should come from registration NCC, not raw keypoint count.


In [ ]:
def build_tissue_mask(img_u8, dilate_px=20):
    nonzero = img_u8[img_u8 > 0]
    if len(nonzero) == 0:
        return np.zeros_like(img_u8)
    thresh, _ = cv2.threshold(nonzero.reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = (img_u8 > thresh).astype(np.uint8) * 255
    if dilate_px > 0:
        kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*dilate_px+1, 2*dilate_px+1))
        mask = cv2.dilate(mask, kern)
    return mask

def count_keypoints(img_u8, threshold=0.0001):
    mask = build_tissue_mask(img_u8)
    detector = cv2.AKAZE_create(threshold=threshold)
    kp, _ = detector.detectAndCompute(img_u8, mask)
    return len(kp), mask

candidates = {
    'CK only (current)':          ck_u8,
    'RGB->gray (untuned)':        rgb_as_gray,
    'Tuned fusion':                tuned,
}

for label, img in candidates.items():
    n_kp, _ = count_keypoints(img)
    print(f"{label:30s} -> {n_kp:5d} tissue-masked AKAZE keypoints")


---
### Next step

Once a weighting looks promising here, the only change needed in
`akaze_tissue_mask_bspline.py` is swapping the channel extraction + `prepare_ck()` call
in `register_slice()` for `prepare_fused()` with the chosen weights — everything else
(tissue masking, AKAZE, B-spline, QC plots) stays as-is.


## 8. All 8 channels encoded into RGB

This is a different problem from the 3-channel case above: 8 channels don't map onto
3 color planes 1:1, so you have to compress 8→3 *before* you can call it "RGB". Two
common ways to do that compression:

**A. Grouped averaging** — manually bucket the 8 channels into 3 biologically-motivated
groups (e.g. structural/nuclear, tumor/epithelial, immune) and average each group into
one plane. Simple, interpretable, weights are yours to pick.

**B. PCA** — treat each pixel as an 8-dim vector across channels, project onto the top
3 principal components, and map those to R/G/B. Data-driven, maximizes variance captured,
but the resulting "channels" are linear combinations with no biological meaning and can
flip sign/look different slice-to-slice if you fit PCA per-slice rather than on a shared
basis.

Same caveat as before applies even harder here: whatever the final RGB composite looks
like, AKAZE will still flatten it to one grayscale plane. So this section is really about
finding the best **8→1 fusion**, with the RGB step just being a visualization waypoint
(or, with grouping, a meaningful single grayscale fusion if you stop at the group-average
stage and skip the final RGB compositing).


In [ ]:
all_channels = arr.astype(np.float32)  # (8, H, W)
n_ch = all_channels.shape[0]
print(f"{n_ch} channels:", CHANNEL_NAMES)

# Per-channel single view, log-normalized, for reference
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(prepare_single(all_channels[i]), cmap='gray')
    ax.set_title(CHANNEL_NAMES[i])
    ax.axis('off')
plt.tight_layout()
plt.show()


### 8A. Grouped averaging -> RGB

Edit `GROUPS` to bucket channel indices into up to 3 groups. Each group is
log-normalized per-channel, averaged, then placed in one R/G/B plane.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Channel -> RGB color LUT (0-255), AF (idx 7) excluded
CHANNEL_COLORS = {
    0: (0,   128, 255),   # DAPI
    1: (51,  255, 51),    # CD31
    2: (255, 51,  51),    # GAP43
    3: (0,   255, 255),   # NFP
    4: (255, 0,   255),   # CD3
    5: (255, 255, 0),     # CD163
    6: (255, 128, 0),     # CK
}

def prepare_single(img_arr, lo=0.1, hi=99.5):
    """Log-normalize a single channel to uint8."""
    img_float = img_arr.astype(np.float32)
    log_img = np.log1p(img_float)
    p_lo, p_hi = np.percentile(log_img[::4, ::4], (lo, hi))
    norm = cv2.normalize(np.clip(log_img, p_lo, p_hi), None, 0, 255, cv2.NORM_MINMAX)
    return norm.astype(np.uint8)

def weighted_avg_color_composite(all_channels, color_lut, lo=0.1, hi=99.5, weights=None):
    """
    Weighted-average blend (bounded brightness, won't wash to white as more
    channels are added) instead of additive/screen blend.
    all_channels: (C, H, W) float array
    color_lut: dict {channel_idx: (R, G, B) in 0-255}
    weights: optional dict {channel_idx: scalar}, default 1.0 for all listed channels
    """
    c, h, w = all_channels.shape
    if weights is None:
        weights = {idx: 1.0 for idx in color_lut}

    acc = np.zeros((h, w, 3), dtype=np.float32)
    total_weight = sum(weights.get(idx, 1.0) for idx in color_lut)

    for idx, color in color_lut.items():
        norm = prepare_single(all_channels[idx], lo, hi).astype(np.float32) / 255.0
        w_ch = weights.get(idx, 1.0)
        color_arr = np.array(color, dtype=np.float32) / 255.0
        acc += (norm[..., None] * color_arr[None, None, :]) * w_ch

    acc = acc / total_weight * 255.0  # normalize by total weight, not channel count alone
    return np.clip(acc, 0, 255).astype(np.uint8)


# Usage
color_composite = weighted_avg_color_composite(all_channels, CHANNEL_COLORS)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(color_composite)
ax.set_title('7 channels (AF excluded), weighted-average color blend')
ax.axis('off')
plt.show()

# What AKAZE would actually see if fed this composite directly
color_gray_flat = cv2.cvtColor(color_composite, cv2.COLOR_RGB2GRAY)

### 8B. PCA -> RGB

Stacks all 8 channels into per-pixel vectors, fits PCA on a (subsampled, for speed)
set of tissue pixels, projects every pixel onto the top 3 components, and rescales
each component to 0-255 for an R/G/B plane.


In [ ]:
from sklearn.decomposition import PCA

def pca_rgb(all_channels, tissue_mask=None, sample_stride=4, n_components=3):
    c, h, w = all_channels.shape
    flat = all_channels.reshape(c, -1).T  # (H*W, C)

    if tissue_mask is not None:
        sample_mask = (tissue_mask.reshape(-1) > 0)
    else:
        sample_mask = np.ones(flat.shape[0], dtype=bool)

    sample = flat[sample_mask][::sample_stride]
    pca = PCA(n_components=n_components)
    pca.fit(sample)
    print('Explained variance ratio:', pca.explained_variance_ratio_)

    proj = pca.transform(flat)  # (H*W, 3)
    rgb = np.zeros((h * w, 3), dtype=np.uint8)
    for k in range(n_components):
        comp = proj[:, k]
        p_lo, p_hi = np.percentile(comp, (0.5, 99.5))
        comp_n = np.clip((comp - p_lo) / max(p_hi - p_lo, 1e-6) * 255, 0, 255)
        rgb[:, k] = comp_n.astype(np.uint8)
    return rgb.reshape(h, w, n_components), pca

# Use the CK tissue mask from earlier as a sampling region so PCA isn't dominated by background
tissue_mask_for_pca = build_tissue_mask(ck_u8)
pca_composite, pca_model = pca_rgb(all_channels, tissue_mask=tissue_mask_for_pca)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(pca_composite)
ax.set_title('PCA(8 channels) -> top 3 components as RGB')
ax.axis('off')
plt.show()

pca_gray_flat = cv2.cvtColor(pca_composite, cv2.COLOR_RGB2GRAY)


### 8C. Compare all 8-channel candidates against the 3-channel baseline

Grayscale flattenings side by side, plus the keypoint-count proxy.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
imgs = {
    'CK only (current)':        ck_u8,
    '3-ch tuned fusion':        tuned,
    'Color-LUT-8ch -> gray':      color_gray_flat,
    'PCA-8ch -> gray':          pca_gray_flat,
}
for ax, (label, img) in zip(axes, imgs.items()):
    ax.imshow(img, cmap='gray')
    ax.set_title(label)
    ax.axis('off')
plt.tight_layout()
plt.show()

for label, img in imgs.items():
    n_kp, _ = count_keypoints(img)
    print(f"{label:25s} -> {n_kp:5d} tissue-masked AKAZE keypoints")


**Reading these results:** PCA components can flip sign or rotate depending on which
pixels you sampled, so don't be surprised if a PCA composite looks visually odd or
low-contrast on first try — that's a sampling/scaling issue, not necessarily a sign
PCA is unsuitable. Grouped averaging is more predictable and easier to defend in a
methods write-up, since each group has a stated biological rationale; PCA may capture
more *variance* but is harder to justify channel-by-channel. If you go down the 8-channel
route at all, grouped averaging collapsed straight to one grayscale fusion (skip the RGB
step entirely) is probably the simplest thing to actually plug into `register_slice()`.
